In [22]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [23]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
82,"The story-line of ""The Thief of Bagdad"" is com...",positive
288,I just got done watching The Edge of Love (by ...,negative
34,Julien Hernandez is certainly an attractive an...,negative
30,CRYSTAL VOYAGER is a strange documentary about...,negative
727,What is supposed to be a simple generic myster...,negative


In [24]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [25]:
df = normalize_text(df)
df.head()

,review,sentiment
82,story line the thief bagdad complex owing told...,positive
288,got done watching edge love by way one worst t...,negative
34,julien hernandez certainly attractive likable ...,negative
30,crystal voyager strange documentary eccentric ...,negative
727,supposed simple generic mystery plot involving...,negative


In [26]:
df['sentiment'].value_counts()

sentiment
negative    256
positive    244
Name: count, dtype: int64

In [27]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [28]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
82,story line the thief bagdad complex owing told...,1
288,got done watching edge love by way one worst t...,0
34,julien hernandez certainly attractive likable ...,0
30,crystal voyager strange documentary eccentric ...,0
727,supposed simple generic mystery plot involving...,0


In [29]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [35]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [37]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/kaleshubham8459/MLOPS---Capstone-Project.mlflow')
dagshub.init(repo_owner='kaleshubham8459', repo_name='YT-Capstone-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-07-08 11:57:42,726 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/kaleshubham8459/YT-Capstone-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "kaleshubham8459/YT-Capstone-Project"

2026-07-08 11:57:43,107 - INFO - Initialized MLflow to track repo "kaleshubham8459/YT-Capstone-Project"


Repository kaleshubham8459/YT-Capstone-Project initialized!

2026-07-08 11:57:43,115 - INFO - Repository kaleshubham8459/YT-Capstone-Project initialized!


<Experiment: artifact_location='mlflow-artifacts:/cafa0861f1e344ff911005a6933e6784', creation_time=1783508330600, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783508330600, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [38]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-08 11:57:45,301 - INFO - Starting MLflow run...
2026-07-08 11:57:46,599 - INFO - Logging preprocessing parameters...
2026-07-08 11:57:47,690 - INFO - Initializing Logistic Regression model...
2026-07-08 11:57:47,949 - INFO - Fitting the model...
2026-07-08 11:57:50,713 - INFO - Model training complete.
2026-07-08 11:57:50,718 - INFO - Logging model parameters...
2026-07-08 11:57:51,019 - INFO - Making predictions...
2026-07-08 11:57:51,030 - INFO - Calculating evaluation metrics...
2026-07-08 11:57:51,195 - INFO - Logging evaluation metrics...
2026-07-08 11:57:52,509 - INFO - Saving and logging the model...
2026/07/08 11:57:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-08 11:58:19,431 - INFO - Model training and logging completed in 32.83 seconds.
2026-07-08 11:58:19,434 - INFO - Accuracy: 0.64
2026-07-08 11:58:19,439 - INFO - Precision: 0.6428571428571429
2026-07-08 11:58:19,443 - INFO - Recall: 0.5625
2026-07-08 11:58:19,445

🏃 View run angry-hog-497 at: https://dagshub.com/kaleshubham8459/MLOPS---Capstone-Project.mlflow/#/experiments/0/runs/9103c3d1557046908c1d010342aea65b
🧪 View experiment at: https://dagshub.com/kaleshubham8459/MLOPS---Capstone-Project.mlflow/#/experiments/0
